# 🚀 PART 9: Performance Optimization & Advanced Operations

# ⚡ Pandas Part 9: Performance Optimization Masterclass
**Vectorization, `eval()`, `query()`, PyArrow, and Out-of-Core Processing**

Welcome to Part 9! When your datasets grow from thousands to millions of rows, standard Pandas operations become too slow. In this module, we transition from writing *working* code to writing *blazing-fast, production-grade* code.

**Framework:** `Theory → Model → Diagram → Code → Real World → Mistakes → Interview → Prediction → Practice (Levels 1-5)`


In [ ]:
# ==========================================
# 🛠️ SETUP: LARGE SCALE MOCK DATA
# ==========================================
import pandas as pd
import numpy as np
import time
import sqlite3

print(f"Pandas Version: {pd.__version__}")

# Generate 2 Million rows for performance testing
np.random.seed(42)
n_rows = 2_000_000
print(f"Generating {n_rows:,} rows...")

large_df = pd.DataFrame({
    'id': range(n_rows),
    'age': np.random.randint(18, 70, n_rows),
    'salary': np.random.randint(30000, 150000, n_rows),
    'department': np.random.choice(['HR', 'IT', 'Sales', 'Marketing'], n_rows),
    'performance_score': np.random.uniform(1.0, 5.0, n_rows).round(2)
})
print("✅ Large Dataset Ready!")


# 📘 Module 1: Vectorization vs. `apply()` (The Golden Rule)

## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** Pandas is built on top of NumPy. NumPy uses **Vectorization** (pushing loops down to optimized C-code). Using Python `for` loops or `.apply()` forces Pandas to step back into slow, unoptimized Python space.
**Mental Model:** 
- **Vectorization:** An automated factory assembly line (processes 1 million items at once).
- **Apply:** A worker picking up one item, processing it, and putting it down, 1 million times.

**Syntax:**
```python
# ❌ SLOW (Python Loop/Apply)
df['new'] = df.apply(lambda row: row['A'] * row['B'], axis=1)

# ✅ FAST (Vectorized)
df['new'] = df['A'] * df['B']
```

## 🌍 8. Real World Example
Calculating a `bonus` column based on `salary` and `performance_score`. Vectorization takes **15ms**. `.apply()` takes **15,000ms** (1000x slower!).

## ⚠️ 9. Common Mistakes
Using `.apply()` for simple math. `.apply()` should *only* be used for complex custom functions that cannot be vectorized (e.g., calling an external API per row).

## 🎤 10. Interview Question
**Q:** When is it acceptable to use `.apply()` in Pandas?
**A:** Only when the operation cannot be vectorized (e.g., complex string parsing, calling a third-party API, or using a custom NLP model per row). For math, logic, and standard string methods, always vectorize.

## 🔮 11. Output Prediction
```python
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print(df.apply(lambda x: x['A'] + x['B'], axis=1).sum())
```
**A:** `21`. (1+4=5, 2+5=7, 3+6=9. 5+7+9 = 21).

## 🎯 12-16. Practice Tasks
*   **Level 1:** Vectorize the creation of a `tax` column (20% of `salary`).
*   **Level 2:** Use `np.where()` to create a `is_senior` boolean column (age > 40).
*   **Level 3:** Vectorize a complex condition: `bonus` is 10% of salary if IT, else 5%.
*   **Level 4:** Time the difference between `.apply()` and Vectorization on `large_df`.
*   **Level 5:** Write a custom function that *requires* `.apply()` (e.g., converting `department` to a specific hash code).


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 1)
# ==========================================
# Level 1
large_df['tax'] = large_df['salary'] * 0.20

# Level 2
large_df['is_senior'] = np.where(large_df['age'] > 40, True, False)

# Level 3
conditions = [large_df['department'] == 'IT', large_df['department'] != 'IT']
choices = [large_df['salary'] * 0.10, large_df['salary'] * 0.05]
large_df['dept_bonus'] = np.select(conditions, choices, default=0)

# Level 5 (Custom Apply)
import hashlib
def get_dept_hash(dept):
    return hashlib.md5(dept.encode()).hexdigest()[:5]

# Only run on a small subset to save time!
large_df['dept_hash'] = large_df['department'].apply(get_dept_hash)
print(large_df[['department', 'dept_hash']].head())
```

</details>


In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 1
# ==========================================
# Example Execution: Vectorization vs Apply
start = time.time()
large_df['bonus_vec'] = large_df['salary'] * 0.10
print(f"Vectorized Time: {time.time() - start:.4f} seconds")

start = time.time()
large_df['bonus_apply'] = large_df.apply(lambda row: row['salary'] * 0.10, axis=1)
print(f"Apply Time: {time.time() - start:.4f} seconds")

# --- YOUR TURN ---
# Level 1: Vectorize tax (20% of salary)
# Level 2: np.where for is_senior
# Level 3: Vectorize department-based bonus
# Level 4: Timing challenge
# Level 5: Custom apply function

# 📘 Module 2: `eval()` and `query()` (Memory & Speed)

## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** When filtering or doing math on large DataFrames, Pandas creates intermediate arrays in memory. `eval()` and `query()` use the `numexpr` library under the hood to evaluate expressions in C, bypassing intermediate memory allocation.
**Mental Model:** 
- **Standard:** Doing math on scratch paper, copying the result, then doing more math.
- **Eval/Query:** Doing the math directly in your head and writing only the final answer.

**Syntax:**
```python
# Standard Filtering (Creates intermediate boolean mask)
df[(df['age'] > 30) & (df['salary'] > 50000)]

# Query (SQL-like, memory efficient)
df.query('age > 30 and salary > 50000')
```

## ⚠️ 9. Common Mistakes
Using Python variables inside `query()` without the `@` symbol.
```python
min_age = 30
df.query('age > @min_age') # ✅ Correct
df.query('age > min_age')  # ❌ KeyError!
```

## 🎤 10. Interview Question
**Q:** Why is `df.query()` faster and more memory-efficient than standard boolean indexing?
**A:** Standard boolean indexing `(df['A'] > 0) & (df['B'] > 0)` creates three separate intermediate arrays in memory (one for A>0, one for B>0, one for the AND operation). `query()` evaluates the entire expression in a single C-level pass, allocating memory only for the final result.

## 🎯 12-16. Practice Tasks
*   **Level 1:** Filter `large_df` for IT department and age > 30 using `.query()`.
*   **Level 2:** Use `@` to filter salary greater than a variable `threshold = 80000`.
*   **Level 3:** Use `eval()` to create a new column: `total_comp = salary + (performance_score * 1000)`.
*   **Level 4:** Chain multiple `.query()` methods to filter complex logic.
*   **Level 5:** Compare the memory usage (`df.memory_usage()`) of standard filtering vs `query()`.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 2)
# ==========================================
# Level 1
print(large_df.query("department == 'IT' and age > 30").head())

# Level 2
threshold = 80000
print(large_df.query("salary > @threshold").head())

# Level 3
large_df.eval("total_comp = salary + (performance_score * 1000)", inplace=True)
print(large_df[['salary', 'performance_score', 'total_comp']].head())

# Level 4
filtered = (large_df.query("department == 'Sales'")
                    .query("age > 25")
                    .query("performance_score > 3.0"))
print(f"Chained filter rows: {len(filtered):,}")

# Level 5
mem_standard = large_df[(large_df['age'] > 30) & (large_df['salary'] > 50000)].memory_usage(deep=True).sum()
mem_query = large_df.query("age > 30 and salary > 50000").memory_usage(deep=True).sum()
print(f"Standard Mem: {mem_standard/1e6:.2f} MB | Query Mem: {mem_query/1e6:.2f} MB")
```

</details>


In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 2
# ==========================================
# Example Execution
senior_it = large_df.query("department == 'IT' and age > 30")
print(f"Senior IT Staff: {len(senior_it):,}")

# --- YOUR TURN ---
# Level 1: Query IT & Age > 30
# Level 2: Query with @ variable
# Level 3: eval() for total_comp
# Level 4: Chained queries
# Level 5: Memory usage comparison

# 📘 Module 3: Pandas 2.0+ PyArrow & Chunking
## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** Pandas 2.0 introduced **PyArrow** as a backend. PyArrow uses columnar memory format (like Apache Arrow), resulting in massive speedups and null-handling improvements. For data larger than RAM, we use **Chunking**.
**Mental Model:** 
- **PyArrow:** Upgrading from a bicycle (NumPy) to a sports car (Apache Arrow).
- **Chunking:** Eating a massive pizza slice-by-slice so you don't choke (Out of Memory).

**Syntax:**
```python
# PyArrow Backend
df = pd.read_csv('data.csv', dtype_backend='pyarrow')

# Chunking
for chunk in pd.read_csv('huge.csv', chunksize=100000):
    process(chunk)
```

## 🎤 10. Interview Question
**Q:** What is the main advantage of the PyArrow backend in Pandas 2.0?
**A:** It provides true nullable data types (no more `Int64` vs `float64` for NaNs), significantly reduces memory footprint via columnar storage, and speeds up I/O and aggregation operations.

## 🎯 12-16. Practice Tasks
*   **Level 1:** Create a small DataFrame using `dtype_backend='pyarrow'`.
*   **Level 2:** Introduce `NaN` values and observe how PyArrow handles them vs standard NumPy.
*   **Level 3:** Write a chunking loop to read a CSV (simulate with `io.StringIO`) and sum a column.
*   **Level 4:** Convert an existing standard DataFrame to PyArrow backend using `.convert_dtypes()`.
*   **Level 5:** Write a function that processes a CSV in chunks and returns the top 5 most frequent categories.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 3)
# ==========================================
# Level 1 & 2
print(df_arrow)
print("Nulls handled natively without upcasting to float!")

# Level 3 (Simulating chunks using StringIO)
import io
csv_data = "id,val\n1,10\n2,20\n3,30\n4,40"
chunks = pd.read_csv(io.StringIO(csv_data), chunksize=2)
total = sum(chunk['val'].sum() for chunk in chunks)
print(f"Chunked Sum: {total}")

# Level 4
std_df = pd.DataFrame({'A': [1, None, 3], 'B': ['a', 'b', None]})
arrow_df = std_df.convert_dtypes(dtype_backend='pyarrow')
print(arrow_df.dtypes)

# Level 5
def get_top_categories(file_path, chunksize=1000):
    # Simulating with our large_df
    counts = {}
    # In reality: for chunk in pd.read_csv(file_path, chunksize=chunksize):
    for i in range(0, len(large_df), chunksize):
        chunk = large_df.iloc[i:i+chunksize]
        for val in chunk['department'].value_counts().items():
            counts[val[0]] = counts.get(val[0], 0) + val[1]
    return pd.Series(counts).nlargest(5)

print(get_top_categories('dummy.csv'))
```

</details>

In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 3
# ==========================================
# Example Execution: PyArrow
df_arrow = pd.DataFrame({
    'A': [1, 2, None],
    'B': ['x', 'y', 'z']
}, dtype_backend='pyarrow')
print(df_arrow.dtypes)

# --- YOUR TURN ---
# Level 1: Create PyArrow DF
# Level 2: NaN handling
# Level 3: Chunking simulation
# Level 4: convert_dtypes()
# Level 5: Chunked frequency counter


# 🏢 PART 10: Data Engineering & ETL Pipelines

# 🔄 Pandas Part 10: Data Engineering & ETL Pipelines
**SQL Integration, API Parsing, and Automated ETL**

Welcome to Part 10! Data doesn't just live in CSVs. In the real world, you must Extract data from SQL databases and APIs, Transform it using Pandas, and Load it into new destinations. This is the **ETL Pipeline**.

**Framework:** `Theory → Model → Diagram → Code → Real World → Mistakes → Interview → Prediction → Practice (Levels 1-5)`


In [ ]:
# ==========================================
# 🛠️ SETUP: SQL & API MOCK ENVIRONMENT
# ==========================================
import pandas as pd
import numpy as np
import sqlite3
import json
import requests
from datetime import datetime

# 1. Create an in-memory SQLite Database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Insert mock employee data
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'David'],
    'dept_id': [101, 102, 101, 103]
})
employees.to_sql('employees', conn, index=False, if_exists='replace')

# Insert mock department data
departments = pd.DataFrame({
    'dept_id': [101, 102, 103],
    'dept_name': ['Engineering', 'Sales', 'HR']
})
departments.to_sql('departments', conn, index=False, if_exists='replace')

# 2. Mock API JSON Response
api_json = {
    "status": "success",
    "data": [
        {"user": {"id": 1, "name": "Alice"}, "metrics": {"sales": 100, "region": "North"}},
        {"user": {"id": 2, "name": "Bob"}, "metrics": {"sales": 150, "region": "South"}}
    ]
}

print("✅ SQL Database & API JSON Ready!")

# 📘 Module 1: SQL Integration (`read_sql` & `to_sql`)

## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** Pandas uses SQLAlchemy (or built-in `sqlite3`) to communicate with relational databases. You can push SQL queries directly into Pandas, and push DataFrames back into SQL tables.
**Mental Model:** 
- **read_sql:** A pipeline bringing water (data) from the reservoir (DB) to your house (Pandas).
- **to_sql:** Pumping treated water back into the reservoir.

**Syntax:**
```python
# Extract
df = pd.read_sql("SELECT * FROM table WHERE condition", con=connection)

# Load
df.to_sql("table_name", con=connection, if_exists="append", index=False)
```

## ⚠️ 9. Common Mistakes
Forgetting `index=False` in `to_sql()`, which creates an ugly, unnamed index column in your database table.

## 🎤 10. Interview Question
**Q:** How do you handle a SQL query that returns 50 million rows?
**A:** You shouldn't load it all into memory at once. Use `chunksize` in `pd.read_sql(query, con, chunksize=100000)` to process the data iteratively, or perform the heavy filtering/aggregation directly in SQL before bringing it into Pandas.

## 🎯 12-16. Practice Tasks
*   **Level 1:** Read the `employees` table into a DataFrame.
*   **Level 2:** Write a SQL query to join `employees` and `departments` and load it into Pandas.
*   **Level 3:** Create a new DataFrame in Pandas and save it to the SQL database as `new_hires`.
*   **Level 4:** Update an existing SQL table by appending new data using `if_exists='append'`.
*   **Level 5:** Write a function that checks if a table exists in the DB, and only creates it if it doesn't.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 1)
# ==========================================
# Level 1
print(pd.read_sql("SELECT * FROM employees", conn))

# Level 2
join_query = """
SELECT e.name, d.dept_name 
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
"""
print(pd.read_sql(join_query, conn))

# Level 3
new_hires = pd.DataFrame({'emp_id': [5], 'name': ['Eve'], 'dept_id': [102]})
new_hires.to_sql('new_hires', conn, index=False, if_exists='replace')

# Level 4
more_hires = pd.DataFrame({'emp_id': [6], 'name': ['Frank'], 'dept_id': [101]})
more_hires.to_sql('new_hires', conn, index=False, if_exists='append')
print(pd.read_sql("SELECT * FROM new_hires", conn))

# Level 5
def create_table_if_not_exists(df, table_name, con):
    # Check if table exists
    query = f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';"
    exists = pd.read_sql(query, con).shape[0] > 0
    
    if not exists:
        df.to_sql(table_name, con, index=False)
        print(f"Table '{table_name}' created.")
    else:
        print(f"Table '{table_name}' already exists.")

create_table_if_not_exists(new_hires, 'new_hires', conn)
```

</details>

In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 1
# ==========================================
# Example Execution
df_emp = pd.read_sql("SELECT * FROM employees", conn)
print(df_emp)

# --- YOUR TURN ---
# Level 1: Read employees
# Level 2: SQL Join via read_sql
# Level 3: to_sql new_hires
# Level 4: Append data
# Level 5: Check table existence


# 📘 Module 2: APIs & `json_normalize()`

## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** APIs return data in nested JSON formats. Standard `pd.DataFrame(json)` fails on nested dictionaries. `pd.json_normalize()` flattens nested JSON into a clean, tabular 2D DataFrame.
**Mental Model:** 
- **Nested JSON:** A set of Russian nesting dolls.
- **json_normalize:** Smashing the dolls open and laying all the pieces flat on a table.

**Syntax:**
```python
# Flatten nested JSON
df = pd.json_normalize(data, record_path=['data'], meta=['status'])
```

## ⚠️ 9. Common Mistakes
Passing a JSON *string* instead of a parsed Python dictionary. You must use `response.json()` or `json.loads()` first.

## 🎤 10. Interview Question
**Q:** How do you handle an API that returns a list of dictionaries where some keys are missing in certain records?
**A:** `pd.json_normalize()` handles this automatically by filling missing keys with `NaN`. Alternatively, `pd.DataFrame.from_records()` does the same.

## 🎯 12-16. Practice Tasks
*   **Level 1:** Parse the `api_json` mock data into a flat DataFrame using `json_normalize`.
*   **Level 2:** Extract the nested `user` and `metrics` into separate column prefixes (e.g., `user_name`, `metrics_sales`).
*   **Level 3:** Handle a deeply nested JSON with a `record_path` (e.g., extracting a list of items inside a receipt).
*   **Level 4:** Simulate an API call using `requests.get()` (use a free public API like `jsonplaceholder.typicode.com/users`).
*   **Level 5:** Write a function that paginates through an API (fetches page 1, 2, 3) and concatenates the results.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 2)
# ==========================================
# Level 1 & 2
df_flat = pd.json_normalize(api_json['data'], sep='_')
print(df_flat.columns) # Shows 'user_name', 'metrics_sales'

# Level 3 (Record Path)
receipt_json = {"store": "Walmart", "items": [{"name": "Apple", "price": 1}, {"name": "Banana", "price": 2}]}
df_items = pd.json_normalize(receipt_json, record_path='items', meta=['store'])
print(df_items)

# Level 4 (Real API)
# response = requests.get('https://jsonplaceholder.typicode.com/users')
# df_users = pd.DataFrame(response.json())
# print(df_users.head())

# Level 5 (Pagination Simulation)
def fetch_paginated_data(base_url, max_pages=3):
    all_data = []
    for page in range(1, max_pages + 1):
        # Simulating API call
        mock_response = [{"page": page, "id": i} for i in range(1, 4)]
        all_data.extend(mock_response)
    return pd.DataFrame(all_data)

print(fetch_paginated_data("dummy_api"))
```

</details>


In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 2
# ==========================================
# Example Execution
df_flat = pd.json_normalize(api_json['data'])
print(df_flat)

# --- YOUR TURN ---
# Level 1: Parse api_json
# Level 2: Max level / separators
# Level 3: record_path challenge
# Level 4: Real API call
# Level 5: Pagination simulation


# 📘 Module 3: Building an Automated ETL Pipeline

## 🧠 1-4. Theory, Model, Diagram & Code
**Theory:** An ETL pipeline consists of **Extract** (getting data), **Transform** (cleaning/shaping), and **Load** (saving to destination). In Pandas, we wrap these steps in robust functions with error handling.
**Mental Model:** 
- **ETL:** A water treatment plant. 
  - *Extract:* Pump water from the river.
  - *Transform:* Filter, purify, and add minerals.
  - *Load:* Pump clean water into the city reservoir.

**Syntax (Pipeline Structure):**
```python
def extract(): ...
def transform(df): ...
def load(df): ...

def run_pipeline():
    raw = extract()
    clean = transform(raw)
    load(clean)
```

## ⚠️ 9. Common Mistakes
Not implementing `try-except` blocks. If the API goes down or the SQL connection drops, the pipeline should log the error and alert the user, not just crash silently.

## 🎤 10. Interview Question
**Q:** How do you ensure idempotency in an ETL pipeline?
**A:** Idempotency means running the pipeline multiple times yields the same result without duplicating data. We achieve this in SQL by using `if_exists='replace'` or using `MERGE`/`UPSERT` statements instead of blind `append`.

## 🎯 12-16. Practice Tasks
*   **Level 1:** Write an `extract()` function that reads from our SQLite DB.
*   **Level 2:** Write a `transform()` function that cleans names (lowercase) and adds a `tax` column.
*   **Level 3:** Write a `load()` function that saves the transformed data to a new SQL table.
*   **Level 4:** Combine them into a `run_etl()` function with a `try-except` block.
*   **Level 5:** Add logging (using Python's `logging` module) to track how many rows were processed.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 3)
# ==========================================
import logging

# Level 5: Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Level 1: Extract
def extract_data(connection):
    logging.info("Extracting data from SQL...")
    query = "SELECT * FROM employees"
    return pd.read_sql(query, connection)

# Level 2: Transform
def transform_data(df):
    logging.info(f"Transforming {len(df)} rows...")
    df = df.copy()
    df['name'] = df['name'].str.lower()
    df['processed_date'] = datetime.now().date()
    return df

# Level 3: Load
def load_data(df, connection, table_name):
    logging.info(f"Loading data into {table_name}...")
    df.to_sql(table_name, connection, if_exists='replace', index=False)
    logging.info("Load complete!")

# Level 4: Run Pipeline
def run_etl_pipeline():
    try:
        raw_data = extract_data(conn)
        if raw_data.empty:
            raise ValueError("Extracted data is empty!")
            
        clean_data = transform_data(raw_data)
        load_data(clean_data, conn, 'employees_cleaned')
        logging.info("ETL Pipeline finished successfully.")
        
    except Exception as e:
        logging.error(f"Pipeline failed: {e}")

# Execute
run_etl_pipeline()
print(pd.read_sql("SELECT * FROM employees_cleaned", conn))
```

</details>


In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 3
# ==========================================
# --- YOUR TURN ---
# Level 1: extract()
# Level 2: transform()
# Level 3: load()
# Level 4: run_etl() with try-except
# Level 5: Add logging


# 🎉 Congratulations! You are now a Pandas Master!
You have completed the entire **Pandas Masterclass Series (Parts 6-10)**.

### 🏆 What You've Conquered:
✅ **Part 6:** MultiIndex, Stack/Unstack, Melt, Pivot Tables
✅ **Part 7:** Time Series Basics, Timestamps, Date Ranges
✅ **Part 8:** Resampling, Rolling Windows, Time Shifts
✅ **Part 9:** Vectorization, `eval()`, PyArrow, Chunking
✅ **Part 10:** SQL Integration, APIs, Automated ETL Pipelines

### 🚀 Next Steps in Your Data Journey:
You now have the Pandas skills of a **Senior Data Engineer**. 
To complete your toolkit, move on to:
1. **Polars / DuckDB:** For out-of-core, multi-threaded data processing.
2. **Airflow / Prefect:** For orchestrating the ETL pipelines you just built.
3. **dbt (Data Build Tool):** For transforming data directly inside SQL warehouses (Snowflake/BigQuery).

*Happy Coding! 🐼✨*